In [68]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity, sigmoid_kernel
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import csr_matrix
import warnings
warnings.filterwarnings('ignore')

### загрузка данных

In [3]:
items = pd.read_csv('/Users/admin/Desktop/RecSys/KION_DATASET/data_original/items.csv')
data_test = pd.read_csv('/Users/admin/Desktop/RecSys/KION_DATASET/test_interactions.csv')
data_train = pd.read_csv('/Users/admin/Desktop/RecSys/KION_DATASET/train_interactions.csv')

### метрики

In [4]:
def precision(recommended_list, bought_list):
    
    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)
    
    flags = np.isin(bought_list, recommended_list)
    
    precision = flags.sum() / len(recommended_list)
    
    return precision


def precision_at_k(recommended_list, bought_list, k=5):
    
    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)
    
    bought_list = bought_list 
    recommended_list = recommended_list[:k]
    
    flags = np.isin(bought_list, recommended_list)
    
    precision = flags.sum() / len(recommended_list)
    
    
    return precision

def ap_k(recommended_list, bought_list, k=5):
    
    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)
    
    flags = np.isin(recommended_list, bought_list)
    
    if sum(flags) == 0:
        return 0
    
    sum_ = 0
    for i in range(0, k-1):
        if flags[i] :
            p_k = precision_at_k(recommended_list, bought_list, k=i+1)
            sum_ += p_k

            
    result = sum_ / sum(flags)
    
    return result

def map_k(recommended_list, bought_list, k=5, u=1):
    
    # your_code
    if u == 1:
        return ap_k(recommended_list[u-1], bought_list[u-1], k=5)
    
    sum = 0
    for i in range(0, u):
        ap_k_map = ap_k(recommended_list[i], bought_list[i], k=5)
        sum += ap_k_map

    result = sum / u
    
    return result

## UserKNN и ItemKNN

In [5]:
class UserKNN:
    def __init__(self, k=3, min_similarity=0.1):

        self.k = k
        self.min_similarity = min_similarity
        self.user_item_matrix = None
        self.user_similarity = None
        self.user_ids = None
        self.item_ids = None
        self.ratings_matrix = None
        
    def fit(self, interactions):

        self.matrix(interactions)
        self.similarity()
        
        return self
    
    def matrix(self, interactions):
        self.user_ids = pd.Categorical(interactions['user_id']).categories
        self.item_ids = pd.Categorical(interactions['item_id']).categories
        
        user_codes = pd.Categorical(interactions['user_id']).codes
        item_codes = pd.Categorical(interactions['item_id']).codes

        self.ratings_matrix = csr_matrix(
            (interactions['watched_pct'], (user_codes, item_codes)),
            shape=(len(self.user_ids), len(self.item_ids))
        )
        self.user_item_matrix = self.ratings_matrix.toarray()
        
        return self.ratings_matrix
    
    def similarity(self):
        self.user_similarity = cosine_similarity(self.user_item_matrix)
        np.fill_diagonal(self.user_similarity, 0)
        
        return self.user_similarity
    
    def user_index(self, user_id):
        if user_id not in self.user_ids:
            return None
        return np.where(self.user_ids == user_id)[0][0]
    
    def item_index(self, item_id):
        if item_id not in self.item_ids:
            return None
        return np.where(self.item_ids == item_id)[0][0]
    
    def similar_users(self, user_id, n_similar=None):
        if n_similar is None:
            n_similar = self.k
            
        user_idx = self.user_index(user_id)
        if user_idx is None:
            return []
        
        user_similarities = self.user_similarity[user_idx]
        similar_indices = np.argsort(user_similarities)[::-1]

        similar_users = []
        for idx in similar_indices:
            similarity = user_similarities[idx]
            if similarity >= self.min_similarity and len(similar_users) < n_similar:
                similar_user_id = self.user_ids[idx]
                similar_users.append((similar_user_id, similarity))
            elif len(similar_users) >= n_similar:
                break
        
        return similar_users
    
    def predict(self, user_id, item_id):
        user_idx = self.user_index(user_id)
        item_idx = self.item_index(item_id)
        
        if user_idx is None or item_idx is None:
            return 0.0

        similar_users = self.similar_users(user_id)
        
        if not similar_users:
            return 0.0

        neighbor_ratings = []
        neighbor_similarities = []
        
        for neighbor_id, similarity in similar_users:
            neighbor_idx = self.user_index(neighbor_id)
            neighbor_rating = self.user_item_matrix[neighbor_idx, item_idx]

            if neighbor_rating > 0:
                neighbor_ratings.append(neighbor_rating)
                neighbor_similarities.append(similarity)

        if not neighbor_ratings:
            return 0.0

        neighbor_ratings = np.array(neighbor_ratings)
        neighbor_similarities = np.array(neighbor_similarities)

        if np.sum(neighbor_similarities) > 0:
            predicted_rating = np.average(neighbor_ratings, weights=neighbor_similarities)
        else:
            predicted_rating = np.mean(neighbor_ratings)
        
        return float(predicted_rating)
    
    def recommend(self, user_id, n_recommendations=10):

        user_idx = self.user_index(user_id)
        if user_idx is None:
            return []

        user_ratings = self.user_item_matrix[user_idx]
        rated_items = set(np.where(user_ratings > 0)[0])

        predictions = []
        for item_idx in range(len(self.item_ids)):
            if item_idx not in rated_items:
                item_id = self.item_ids[item_idx]
                predicted_rating = self.predict(user_id, item_id)
                
                if predicted_rating > 0:
                    predictions.append((item_id, predicted_rating))

        predictions.sort(key=lambda x: x[1], reverse=True)
        
        return [item_id for item_id, _ in predictions[:n_recommendations]]

In [6]:
class ItemKNN:
    def __init__(self, k=3, min_similarity=0.1):

        self.k = k
        self.min_similarity = min_similarity
        self.user_item_matrix = None
        self.item_similarity = None
        self.user_ids = None
        self.item_ids = None
        self.ratings_matrix = None
        
    def fit(self, interactions):

        self.matrix(interactions)
        self.similarity()
        
        return self
    
    def matrix(self, interactions):
        self.user_ids = pd.Categorical(interactions['user_id']).categories
        self.item_ids = pd.Categorical(interactions['item_id']).categories
        
        user_codes = pd.Categorical(interactions['user_id']).codes
        item_codes = pd.Categorical(interactions['item_id']).codes

        self.ratings_matrix = csr_matrix(
            (interactions['watched_pct'], (user_codes, item_codes)),
            shape=(len(self.user_ids), len(self.item_ids))
        )
    
        self.user_item_matrix = self.ratings_matrix.toarray()
        
        return self.ratings_matrix
    
    def similarity(self):
        item_matrix = self.ratings_matrix.T
        self.item_similarity = cosine_similarity(item_matrix)
        np.fill_diagonal(self.item_similarity, 0)
        
        return self.item_similarity
    
    def user_index(self, user_id):
        if user_id not in self.user_ids:
            return None
        return np.where(self.user_ids == user_id)[0][0]
    
    def item_index(self, item_id):
        if item_id not in self.item_ids:
            return None
        return np.where(self.item_ids == item_id)[0][0]
    
    def similar_items(self, item_id, n_similar=None):
        if n_similar is None:
            n_similar = self.k
            
        item_idx = self.item_index(item_id)
        if item_idx is None:
            return []

        item_similarities = self.item_similarity[item_idx]

        similar_indices = np.argsort(item_similarities)[::-1]
   
        similar_items = []
        for idx in similar_indices:
            similarity = item_similarities[idx]
            if similarity >= self.min_similarity and len(similar_items) < n_similar:
                similar_item_id = self.item_ids[idx]
                similar_items.append((similar_item_id, similarity))
            elif len(similar_items) >= n_similar:
                break
        
        return similar_items
    
    def predict(self, user_id, item_id):

        user_idx = self.user_index(user_id)
        item_idx = self.item_index(item_id)
        
        if user_idx is None or item_idx is None:
            return 0.0

        similar_items = self.similar_items(item_id)
        
        if not similar_items:
            return 0.0

        neighbor_ratings = []
        neighbor_similarities = []
        
        for similar_item_id, similarity in similar_items:
            similar_item_idx = self.item_index(similar_item_id)
            user_rating = self.user_item_matrix[user_idx, similar_item_idx]

            if user_rating > 0:
                neighbor_ratings.append(user_rating)
                neighbor_similarities.append(similarity)

        if not neighbor_ratings:
            return 0.0

        neighbor_ratings = np.array(neighbor_ratings)
        neighbor_similarities = np.array(neighbor_similarities)

        if np.sum(neighbor_similarities) > 0:
            predicted_rating = np.average(neighbor_ratings, weights=neighbor_similarities)
        else:
            predicted_rating = np.mean(neighbor_ratings)
        
        return float(predicted_rating)
    
    def recommend(self, user_id, n_recommendations=10):

        user_idx = self.user_index(user_id)
        if user_idx is None:
            return []

        user_ratings = self.user_item_matrix[user_idx]
        rated_items = set(np.where(user_ratings > 0)[0])

        predictions = []
        for item_idx in range(len(self.item_ids)):
            if item_idx not in rated_items:
                item_id = self.item_ids[item_idx]
                predicted_rating = self.predict(user_id, item_id)
                
                if predicted_rating > 0:
                    predictions.append((item_id, predicted_rating))

        predictions.sort(key=lambda x: x[1], reverse=True)
        
        return [item_id for item_id, _ in predictions[:n_recommendations]]

In [7]:
model_UserKNN = UserKNN(k=5, min_similarity=0.07)
model_UserKNN.fit(data_train)

In [8]:
model_ItemKNN = ItemKNN(k=9, min_similarity=0.1)
model_ItemKNN.fit(data_train)

### предсказания моделей

In [9]:
data_test_group = data_test.groupby(data_test['user_id'])['item_id'].unique().reset_index()
data_test_group.columns = ['user_id', 'item_id']
data_test_group

,user_id,item_id
0,2756,"[1247, 2925, 13325, 7793, 15399, 14488]"
1,4027,"[5732, 3838, 13159, 3349]"
2,4249,"[13483, 9778, 10125, 2865]"
3,4997,"[3190, 5732, 9194]"
4,5119,"[12356, 13849, 2596, 7968]"
...,...,...
1115,1093424,"[10636, 14488, 11118]"
1116,1094628,"[13543, 125, 10119, 9194, 3518]"
1117,1095418,"[14317, 2025, 1290]"
1118,1097444,"[13650, 12841, 12250, 2483]"


In [10]:
userids = data_test_group['user_id'].values
 
userids_test = np.arange(len(userids))

In [11]:
recos_user = []
for i in range(len(userids_test)):
   recommendations = model_UserKNN.recommend(userids[i], n_recommendations=10)
   recos_user.append({'item_id': recommendations})

result_user_knn = pd.DataFrame(recos_user, userids).reset_index()
result_user_knn


,index,item_id
0,2756,"[6948, 9731, 14431, 7019, 13855, 14717, 7731, ..."
1,4027,"[456, 1132, 1313, 2852, 4409, 5906, 8727, 2220..."
2,4249,"[2741, 3475, 3734, 4436, 7417, 7597, 9550, 100..."
3,4997,"[565, 3182, 3571, 3834, 4845, 14177, 14317, 15..."
4,5119,"[3784, 4952, 6162, 6835, 7545, 8636, 9728, 100..."
...,...,...
1115,1093424,"[4689, 1998, 5250, 12974, 10440, 7019, 14213, ..."
1116,1094628,"[9731, 10077, 13865, 14431, 273, 741, 3978, 44..."
1117,1095418,"[5109, 5755, 14563, 334, 1000, 3130, 3259, 373..."
1118,1097444,"[1995, 4635, 5964, 9728, 10469, 11112, 11469, ..."


In [12]:
recos_item = []
for i in range(len(userids_test)):
   recommendations = model_ItemKNN.recommend(userids[i], n_recommendations=10)
   recos_item.append({'item_id': recommendations})

result_item_knn = pd.DataFrame(recos_item, userids).reset_index()
result_item_knn

,index,item_id
0,2756,"[12659, 2798, 3320, 3587, 3873, 4471, 4515, 49..."
1,4027,"[456, 1313, 2852, 4409, 4662, 8727, 9623, 834,..."
2,4249,"[758, 1527, 1569, 3791, 7425, 7571, 11564, 135..."
3,4997,"[1181, 2618, 2626, 3571, 7134, 9885, 10726, 11..."
4,5119,"[2293, 2296, 2373, 2758, 3385, 3905, 4029, 495..."
...,...,...
1115,1093424,"[383, 1590, 2027, 2981, 3591, 4706, 5291, 6626..."
1116,1094628,"[2258, 273, 1508, 3402, 3734, 5644, 5869, 6171..."
1117,1095418,"[1443, 11239, 241, 349, 582, 758, 843, 1527, 1..."
1118,1097444,"[281, 1594, 1995, 2892, 3208, 3682, 3696, 3940..."


In [13]:
print('UserKNN_map_k =', map_k(result_user_knn['item_id'], data_test_group['item_id'], k=10, u=len(data_test_group)))

UserKNN_map_k = 0.020312499999999997


In [14]:
print('ItemKNN_map_k =', map_k(result_item_knn['item_id'], data_test_group['item_id'], k=10, u=len(data_test_group)))

ItemKNN_map_k = 0.0023809523809523807


### рекомендации для случайного пользователя

In [55]:
random_user = np.random.choice(data_test['user_id'], replace=False)

In [17]:
data_train_group = data_train.groupby(data_train['user_id'])['item_id'].unique().reset_index()
data_train_group.columns = ['user_id', 'item_id']

In [56]:
rec_user = result_user_knn.loc[result_user_knn['index']==random_user, 'item_id'].iloc[0]
rec_item = result_item_knn.loc[result_item_knn['index']==random_user, 'item_id'].iloc[0]
rec_test = data_test_group.loc[data_test_group['user_id']==random_user, 'item_id'].iloc[0]
rec_train = data_train_group.loc[data_train_group['user_id']==random_user, 'item_id'].iloc[0]

pred_user = []
test = []
pred_item = []
train = []


for i in range(len(rec_user)):
    title_item = items.loc[items['item_id']==rec_user[i], 'title'].iloc[0]
    pred_user.append(title_item)

for i in range(len(rec_item)):
    title_item = items.loc[items['item_id']==rec_item[i], 'title'].iloc[0]
    pred_item.append(title_item)

for i in range(len(rec_test)):
    title_item = items.loc[items['item_id']==rec_test[i], 'title'].iloc[0]
    test.append(title_item)

for i in range(len(rec_train)):
    title_item = items.loc[items['item_id']==rec_train[i], 'title'].iloc[0]
    train.append(title_item)

print(f"Рекомендаций для пользователя {random_user}:") 
print('UserKNN')
for i, (item) in enumerate(zip(pred_user)):
    print(f"{i+1}. {item}")
print('ItemKNN')
for i, (item) in enumerate(zip(pred_item)):
    print(f"{i+1}. {item}")
print('Test')
for i, (item) in enumerate(zip(test)):
    print(f"{i+1}. {item}")
print('Train')
for i, (item) in enumerate(zip(train)):
    print(f"{i+1}. {item}")

Рекомендаций для пользователя 274701:
UserKNN
1. ('Зверополис',)
2. ('Непосредственно Каха',)
3. ('Алита: Боевой ангел',)
4. ('100% волк',)
5. ('В поисках Немо',)
6. ('Университет монстров',)
7. ('В поисках Дори',)
8. ('Гадкий я',)
9. ('История игрушек 4',)
10. ('Рататуй',)
ItemKNN
1. ('Госпожа Бовари',)
2. ('Веселая ферма',)
3. ('Купи меня',)
4. ('Под водой',)
5. ('Катя и Эф. Куда-Угодно-Дверь',)
6. ('Приключения Электроника',)
7. ('Свой среди чужих, чужой среди своих',)
8. ('Джунгли',)
9. ('Игры судьбы',)
10. ('Гипер-интеллект',)
Test
1. ('Пила 8',)
2. ('Алита: Боевой ангел',)
3. ('Радиовспышка',)
4. ('Дом странных детей Мисс Перегрин',)
5. ('Зверополис',)
Train
1. ('Корпорация монстров',)
2. ('Суперсемейка',)
3. ('Тайна Коко',)
4. ('Малыш на драйве',)
